In [2]:
!pip install tensorflow-model-optimization tf_keras

     ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
     -- ------------------------------------- 0.1/1.7 MB 3.2 MB/s eta 0:00:01
     ------ --------------------------------- 0.3/1.7 MB 4.2 MB/s eta 0:00:01
     ------- -------------------------------- 0.3/1.7 MB 3.4 MB/s eta 0:00:01
     ---------- ----------------------------- 0.5/1.7 MB 3.1 MB/s eta 0:00:01
     -------------- ------------------------- 0.6/1.7 MB 3.3 MB/s eta 0:00:01
     ------------------ --------------------- 0.8/1.7 MB 3.5 MB/s eta 0:00:01
     ---------------------- ----------------- 0.9/1.7 MB 3.5 MB/s eta 0:00:01
     ------------------------ --------------- 1.1/1.7 MB 3.5 MB/s eta 0:00:01
     ---------------------------- ----------- 1.2/1.7 MB 3.5 MB/s eta 0:00:01
     ------------------------------ --------- 1.3/1.7 MB 3.4 MB/s eta 0:00:01
     ------------------------------ --------- 1.3/1.7 MB 3.4 MB/s eta 0:00:01
     ---------------------------------- ----- 1.5/1.7 MB 3.2 MB/s eta 0

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'D:\\bintang\\.venv\\Lib\\site-packages\\google\\~-otobuf\\internal\\_api_implementation.cp310-win_amd64.pyd'
Check the permissions.


[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ============================================================================
# IMPORTS — QAT REQUIRES tf_keras (LEGACY Keras 2.x)
# ============================================================================
# tensorflow_model_optimization tidak support Keras 3 (default di TF 2.16+).
# Solusinya: pakai legacy Keras 2.x via paket `tf_keras` dan set env var.
# ENV VAR HARUS DI-SET SEBELUM `import tensorflow`.
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import time, gzip, shutil, json, tempfile
import multiprocessing
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf

# Pakai tf_keras (legacy) untuk semua layer & model
import tf_keras
from tf_keras.models import Model
from tf_keras.layers import (Layer, Conv2D, MaxPooling2D, Flatten, Dense,
                             GlobalAveragePooling2D, Reshape, Multiply, Input)
from tf_keras.callbacks import EarlyStopping
from tf_keras.utils import to_categorical

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

try:
    import tensorflow_model_optimization as tfmot
    print(f"tfmot version: {tfmot.__version__}")
except ImportError:
    raise ImportError(
        "tensorflow_model_optimization belum terinstal. Jalankan:\n"
        "    pip install tensorflow-model-optimization tf_keras"
    )

print(f"TF version       : {tf.__version__}")
print(f"tf_keras version : {tf_keras.__version__}")

# ----------------------------------------------------------------------------
# Custom ChannelAttention — pakai tf_keras Layer, bukan tf.keras Layer
# ----------------------------------------------------------------------------
class ChannelAttention(Layer):
    """Channel Attention Module — pakai tf_keras (legacy)."""
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):
        channels      = input_shape[-1]
        self.gap      = GlobalAveragePooling2D()
        self.dense1   = Dense(max(1, channels // self.ratio), activation="relu")
        self.dense2   = Dense(channels, activation="sigmoid")
        self.reshape  = Reshape((1, 1, channels))
        super().build(input_shape)

    def call(self, x):
        attn = self.gap(x)
        attn = self.dense1(attn)
        attn = self.dense2(attn)
        attn = self.reshape(attn)
        return Multiply()([x, attn])

    def get_config(self):
        config = super().get_config()
        config.update({"ratio": self.ratio})
        return config

CUSTOM_OBJECTS = {"ChannelAttention": ChannelAttention}



tfmot version: 0.8.0
TF version       : 2.21.0
tf_keras version : 2.21.0


In [ ]:
# ============================================================================
# UNIFIED EVALUATION HELPERS  (sama dengan notebook lain)
# ============================================================================
def get_file_size_kb(path):
    return os.path.getsize(path) / 1024.0 if os.path.exists(path) else 0.0


def evaluate_pipeline(extractor, svm_clf, scaler, X_test, y_test_int,
                      class_names, label="Model", warmup=True):
    if warmup:
        try: _ = extractor.predict(X_test[:2], verbose=0)
        except Exception: pass

    t0 = time.perf_counter()
    X_test_feat = extractor.predict(X_test, verbose=0)
    feat_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    X_test_scaled = scaler.transform(X_test_feat)
    y_pred        = svm_clf.predict(X_test_scaled)
    y_pred_proba  = svm_clf.predict_proba(X_test_scaled)
    inf_time      = time.perf_counter() - t0

    n_classes = len(class_names)
    cm = confusion_matrix(y_test_int, y_pred, labels=list(range(n_classes)))

    sens, spec, prec, f1s = [], [], [], []
    for i in range(n_classes):
        TP = cm[i, i]; FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP; TN = cm.sum() - TP - FN - FP
        s  = TP/(TP+FN) if (TP+FN)>0 else 0.0
        sp = TN/(TN+FP) if (TN+FP)>0 else 0.0
        p  = TP/(TP+FP) if (TP+FP)>0 else 0.0
        f  = 2*p*s/(p+s) if (p+s)>0 else 0.0
        sens.append(s); spec.append(sp); prec.append(p); f1s.append(f)

    y_test_bin   = label_binarize(y_test_int, classes=list(range(n_classes)))
    auc_per_cls  = roc_auc_score(y_test_bin, y_pred_proba, average=None , multi_class="ovr")
    auc_macro    = roc_auc_score(y_test_bin, y_pred_proba, average="macro", multi_class="ovr")
    auc_micro    = roc_auc_score(y_test_bin, y_pred_proba, average="micro", multi_class="ovr")

    return {
        "label": label,
        "accuracy"     : accuracy_score(y_test_int, y_pred),
        "sensitivity"  : list(sens), "specificity": list(spec),
        "precision"    : list(prec), "f1": list(f1s),
        "auc_per_class": list(auc_per_cls),
        "auc_macro"    : auc_macro, "auc_micro": auc_micro,
        "cm"           : cm,
        "feat_time"    : feat_time, "inf_time": inf_time,
        "y_pred"       : y_pred, "y_pred_proba": y_pred_proba,
    }


def per_class_metrics_df(metrics, class_names):
    rows = []
    rows.append(["Sensitivity"] + [f"{v:.4f}" for v in metrics["sensitivity" ]] + [f'{np.mean(metrics["sensitivity" ]):.4f}'])
    rows.append(["Specificity"] + [f"{v:.4f}" for v in metrics["specificity" ]] + [f'{np.mean(metrics["specificity" ]):.4f}'])
    rows.append(["Precision"  ] + [f"{v:.4f}" for v in metrics["precision"   ]] + [f'{np.mean(metrics["precision"   ]):.4f}'])
    rows.append(["F1-Score"   ] + [f"{v:.4f}" for v in metrics["f1"          ]] + [f'{np.mean(metrics["f1"          ]):.4f}'])
    rows.append(["AUC"        ] + [f"{v:.4f}" for v in metrics["auc_per_class"]] + [f'{np.mean(metrics["auc_per_class"]):.4f}'])
    return pd.DataFrame(rows, columns=["Metric"] + class_names + ["Mean"])


def print_evaluation_block(metrics, class_names, title="EVALUATION METRICS"):
    print("=" * 78)
    print(f"  {title} — {metrics['label']}")
    print("=" * 78)
    print(per_class_metrics_df(metrics, class_names).to_string(index=False))
    print("-" * 78)
    print(f"Accuracy        : {metrics['accuracy' ]:.4f}")
    print(f"AUC (macro avg) : {metrics['auc_macro']:.4f}")
    print(f"AUC (micro avg) : {metrics['auc_micro']:.4f}")
    print("\nConfusion Matrix (rows=true, cols=pred):")
    cm_df = pd.DataFrame(metrics["cm"], index=class_names, columns=class_names)
    print(cm_df.to_string()); print("=" * 78)


def resource_table(rows):
    return pd.DataFrame(rows, columns=["Model","Size (KB)","Parameters",
                                       "Feature Extraction (s)","Inference (s)"])


def plot_confusion_matrix(metrics, class_names, ax=None, title=None):
    if ax is None: fig, ax = plt.subplots(figsize=(5,4))
    sns.heatmap(metrics["cm"], annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title or f"Confusion Matrix — {metrics['label']}")
    return ax


# ============================================================================
# REPRODUCIBILITY
# ============================================================================
import random
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for g in gpus:
        try: tf.config.experimental.set_memory_growth(g, True)
        except RuntimeError: pass


# ============================================================================
# DATASET LOADER
# ============================================================================
dataset_base_path = "./dataset_processed2"
img_size          = 224
categories        = ["Bengin cases", "Malignant cases", "Normal cases"]
class_names       = ["Bengin", "Malignant", "Normal"]
num_classes       = len(categories)

def load_split_data(split_path, categories):
    X, y = [], []
    for class_idx, cat in enumerate(categories):
        cat_path = os.path.join(split_path, cat)
        if not os.path.isdir(cat_path): continue
        for fn in sorted(os.listdir(cat_path)):
            if not fn.lower().endswith((".jpg", ".jpeg", ".png")): continue
            img = cv2.imread(os.path.join(cat_path, fn))
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (img_size, img_size))
            img = img.astype(np.float32) / 255.0
            X.append(img); y.append(class_idx)
    return np.array(X, dtype=np.float32), np.array(y)

print("Loading dataset…")
X_train, y_train_labels = load_split_data(os.path.join(dataset_base_path, "train"), categories)
X_valid, y_valid_labels = load_split_data(os.path.join(dataset_base_path, "valid"), categories)
X_test , y_test_labels  = load_split_data(os.path.join(dataset_base_path, "test" ), categories)

y_train     = to_categorical(y_train_labels, num_classes=num_classes)
y_valid     = to_categorical(y_valid_labels, num_classes=num_classes)
y_test      = to_categorical(y_test_labels , num_classes=num_classes)
y_train_int = y_train_labels
y_valid_int = y_valid_labels
y_test_int  = y_test_labels
print(f"Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}")


# ============================================================================
# LOAD ORIGINAL ARTEFACTS
# ============================================================================
ORIG_FOLDER = "saved_models_original"
if not os.path.isdir(ORIG_FOLDER):
    raise FileNotFoundError(
        f"Folder '{ORIG_FOLDER}' tidak ditemukan. "
        "Jalankan training.ipynb lebih dulu."
    )

cnn_path       = os.path.join(ORIG_FOLDER, "cnn_attention_model.keras")
extractor_path = os.path.join(ORIG_FOLDER, "feature_extractor.keras")
svm_path       = os.path.join(ORIG_FOLDER, "svm_classifier.pkl")
scaler_path    = os.path.join(ORIG_FOLDER, "feature_scaler.pkl")

print(f"\nLoading artefacts dari {ORIG_FOLDER}/ …")

# ============================================================================
# LOAD MODEL DENGAN tf_keras.models.load_model() (simple & reliable)
# ============================================================================
def _build_arch_tfkeras(input_shape, num_classes):
    """Bangun arsitektur identik training.ipynb dengan tf_keras."""
    inputs = Input(shape=input_shape)
    x = Conv2D(64,(3,3),activation="relu",padding="same")(inputs)
    x = ChannelAttention(ratio=8)(x); x = MaxPooling2D((2,2))(x)
    x = Conv2D(64,(3,3),activation="relu",padding="same")(x)
    x = ChannelAttention(ratio=8)(x); x = MaxPooling2D((2,2))(x)
    x = Flatten()(x)
    feat = Dense(16, activation="relu", name="feature_layer")(x)
    out  = Dense(num_classes, activation="softmax", name="softmax_output")(feat)
    return Model(inputs, out, name="CNN_Attention_QAT")


print("Step 1: Attempt to load model dengan tf_keras.models.load_model()…")
try:
    model_orig = tf_keras.models.load_model(
        cnn_path, 
        custom_objects={"ChannelAttention": ChannelAttention}
    )
    print("  ✓ Model loaded berhasil dari .keras file")
except Exception as e:
    print(f"  ✗ Load gagal ({type(e).__name__}: {e})")
    print("  → Fallback: Rebuild arsitektur + manual weight loading…")
    
    # Fallback: bangun arsitektur baru dan load bobot dari file H5 dalam archive
    model_orig = _build_arch_tfkeras(X_train.shape[1:], num_classes)
    _ = model_orig(X_train[:1])
    
    try:
        import zipfile, tempfile
        with zipfile.ZipFile(cnn_path) as z, tempfile.TemporaryDirectory() as td:
            z.extractall(td)
            h5_inner = os.path.join(td, "model.weights.h5")
            if not os.path.exists(h5_inner):
                for root, _, files in os.walk(td):
                    for f in files:
                        if f.endswith(".h5"):
                            h5_inner = os.path.join(root, f)
                            break
            if os.path.exists(h5_inner):
                model_orig.load_weights(h5_inner)
                print("  ✓ Bobot loaded dari extracted .h5")
    except Exception as e2:
        print(f"  ✗ Fallback juga gagal: {e2}")
        raise

print(f"  Architecture params: {model_orig.count_params():,}")

extractor_orig = Model(model_orig.input,
                       model_orig.get_layer("feature_layer").output,
                       name="feature_extractor")

# Compile model_orig agar bisa di-evaluasi
model_orig.compile(optimizer="adam",
                   loss="categorical_crossentropy", metrics=["accuracy"])

svm_orig    = joblib.load(svm_path)
scaler_orig = joblib.load(scaler_path)
orig_size_cnn       = get_file_size_kb(cnn_path)
orig_size_extractor = get_file_size_kb(extractor_path)
orig_size_svm       = get_file_size_kb(svm_path)
orig_size_scaler    = get_file_size_kb(scaler_path)

print(f"\nCNN params       : {model_orig.count_params():,}")
print(f"Extractor params : {extractor_orig.count_params():,}")
print(f"CNN size on disk : {orig_size_cnn:.2f} KB")


Loading dataset…
Train: (737, 224, 224, 3), Valid: (158, 224, 224, 3), Test: (159, 224, 224, 3)

Loading artefacts dari saved_models_original/ …
Step 1: Build arsitektur dengan tf_keras…


  Architecture params: 3,252,243
Step 2: Load bobot dari file Keras 3 (.keras zip → .weights.h5)…


BadZipFile: File is not a zip file

In [ ]:
# ============================================================================
# PIPELINE A: ORIGINAL  (no retraining)
# ============================================================================
metrics_orig = evaluate_pipeline(
    extractor_orig, svm_orig, scaler_orig,
    X_test, y_test_int, class_names, label="Original (loaded)"
)
print_evaluation_block(metrics_orig, class_names, "ORIGINAL — TEST SET")


# ============================================================================
# PIPELINE B: QUANTIZATION-AWARE TRAINING (QAT)
# ============================================================================
print("\n" + "=" * 78)
print("BUILDING QAT MODEL (selective annotation) …")
print("=" * 78)

# ▶ PARAMETER YANG BISA DIUBAH
QAT_FINETUNE_EPOCHS = 5     # epoch untuk fine-tune dengan fake quantization
QAT_BATCH_SIZE      = 8
QAT_LEARNING_RATE   = 1e-4  # learning rate kecil — kita hanya menyesuaikan, bukan training dari nol


# ----------------------------------------------------------------------------
# Selective annotation: hanya layer standar (Conv2D, Dense, MaxPool, Flatten)
# yang di-quantize. Custom ChannelAttention TIDAK di-quantize karena tfmot
# tidak punya QuantizeConfig untuknya — ini adalah trade-off yang umum
# untuk QAT pada model dengan custom layer.
# ----------------------------------------------------------------------------
quantize_annotate_layer = tfmot.quantization.keras.quantize_annotate_layer
quantize_apply          = tfmot.quantization.keras.quantize_apply
quantize_scope          = tfmot.quantization.keras.quantize_scope

QAT_QUANTIZABLE = (Conv2D, Dense, MaxPooling2D, Flatten)

def apply_qat_annotation(layer):
    """Bungkus layer dengan quantize_annotate_layer jika tipenya supported."""
    if isinstance(layer, QAT_QUANTIZABLE):
        return quantize_annotate_layer(layer)
    return layer

# Clone model dengan annotation per-layer
annotated_model = tf_keras.models.clone_model(
    model_orig, clone_function=apply_qat_annotation
)

# Apply quantization — butuh quantize_scope agar custom layer dikenali
with quantize_scope({"ChannelAttention": ChannelAttention}):
    qat_model = quantize_apply(annotated_model)

qat_model.compile(
    optimizer=tf_keras.optimizers.Adam(learning_rate=QAT_LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
print(f"QAT model dibuat — params: {qat_model.count_params():,}")
print(f"(parameter naik sedikit karena ada 'fake quantization' nodes tambahan)")
qat_model.summary(line_length=120)


# ----------------------------------------------------------------------------
# Fine-tune QAT model — di sini lah model belajar untuk robust ke INT8
# ----------------------------------------------------------------------------
print("\n" + "-" * 78)
print(f"Fine-tuning QAT model selama {QAT_FINETUNE_EPOCHS} epoch "
      f"(LR={QAT_LEARNING_RATE}) …")
print("-" * 78)

t0 = time.perf_counter()
qat_history = qat_model.fit(
    X_train, y_train,
    epochs          = QAT_FINETUNE_EPOCHS,
    batch_size      = QAT_BATCH_SIZE,
    validation_data = (X_valid, y_valid),
    callbacks       = [EarlyStopping(patience=2, monitor="val_loss",
                                     restore_best_weights=True)],
    verbose         = 1,
)
qat_finetune_time = time.perf_counter() - t0
print(f"\nFine-tune time: {qat_finetune_time:.3f} s")


# ----------------------------------------------------------------------------
# Convert QAT model → TFLite INT8
# Karena model sudah dilatih dengan fake quantization, konversi ke INT8
# nyaris lossless (akurasi turun jauh lebih sedikit dibanding PTQ INT8).
# ----------------------------------------------------------------------------
print("\n" + "-" * 78)
print("Converting QAT model → TFLite INT8 …")
print("-" * 78)

os.makedirs("artifacts", exist_ok=True)
qat_tflite_path = "artifacts/qat_int8.tflite"

# Buat feature extractor dari QAT model (sebelum konversi TFLite)
# Setelah quantize_apply, nama layer berubah dari 'feature_layer' menjadi
# 'quant_feature_layer' (Conv2D, Dense, dll. di-wrap dalam QuantizeWrapperV2).
def _find_feature_layer(qmodel, target_name="feature_layer"):
    """Cari layer feature_layer di QAT model — bisa bernama feature_layer
    atau quant_feature_layer tergantung apakah di-wrap quantize_apply."""
    for cand in (target_name, f"quant_{target_name}"):
        try:
            return qmodel.get_layer(cand)
        except ValueError:
            continue
    raise ValueError(
        f"Tidak ditemukan layer '{target_name}' atau 'quant_{target_name}' "
        f"di QAT model. Layer yang ada: {[l.name for l in qmodel.layers]}")

qat_feature_layer = _find_feature_layer(qat_model, "feature_layer")
qat_extractor_keras = Model(
    qat_model.input,
    qat_feature_layer.output,
)

# Konversi ke TFLite via concrete-function (workaround Keras 3 / TF 2.18 bug)
def _make_concrete_fn(keras_model):
    in_shape = tuple(keras_model.input_shape)
    spec = tf.TensorSpec(shape=in_shape, dtype=tf.float32, name="input")
    class _Wrap(tf.Module):
        def __init__(self, m):
            super().__init__(); self.m = m
        @tf.function(input_signature=[spec])
        def __call__(self, x):
            return self.m(x, training=False)
    wrapper  = _Wrap(keras_model)
    concrete = wrapper.__call__.get_concrete_function()
    return concrete, wrapper

t0 = time.perf_counter()
# QAT model TETAP butuh representative_dataset untuk full INT8 — bukan untuk
# kalibrasi internal (sudah dilakukan saat fine-tuning), tapi untuk menentukan
# scale/zero_point input dan output. Karena sudah QAT, beberapa sampel cukup.
QAT_REPR_SAMPLES = 10

def _qat_representative_dataset():
    idxs = np.random.RandomState(42).choice(len(X_train),
                                            size=min(QAT_REPR_SAMPLES, len(X_train)),
                                            replace=False)
    for j, i in enumerate(idxs):
        if j % 3 == 0:
            print(f"  kalibrasi sampel {j+1}/{QAT_REPR_SAMPLES} …", flush=True)
        yield [X_train[i:i+1].astype(np.float32)]

concrete, wrapper = _make_concrete_fn(qat_extractor_keras)
try:
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete], wrapper)
except TypeError:
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete])
converter.optimizations          = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = _qat_representative_dataset
try:
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
        tf.lite.OpsSet.TFLITE_BUILTINS,
    ]
    qat_tflite = converter.convert()
    qat_mode   = "qat-int8"
except Exception as e:
    print(f"[QAT→TFLite] Full INT8 gagal: {e}")
    print("[QAT→TFLite] Fallback ke dynamic-range …")
    concrete, wrapper = _make_concrete_fn(qat_extractor_keras)
    try:
        converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete], wrapper)
    except TypeError:
        converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete])
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    qat_tflite = converter.convert()
    qat_mode   = "qat-dynamic"

with open(qat_tflite_path, "wb") as f:
    f.write(qat_tflite)
qat_convert_time = time.perf_counter() - t0
print(f"QAT → TFLite konversi: {qat_convert_time:.3f} s  (mode = {qat_mode})")
print(f"QAT TFLite ukuran   : {get_file_size_kb(qat_tflite_path):.2f} KB  "
      f"({get_file_size_kb(extractor_path)/get_file_size_kb(qat_tflite_path):.1f}x lebih kecil)")


# ----------------------------------------------------------------------------
# TFLite Feature Extractor wrapper (sama dengan di quantization.ipynb)
# ----------------------------------------------------------------------------
class TFLiteFeatureExtractor:
    def __init__(self, tflite_path, num_threads=None):
        self.path = tflite_path
        n_threads = num_threads or max(1, multiprocessing.cpu_count() - 1)
        self.interpreter = tf.lite.Interpreter(model_path=tflite_path,
                                               num_threads=n_threads)
        self.interpreter.allocate_tensors()
        in_d  = self.interpreter.get_input_details()[0]
        out_d = self.interpreter.get_output_details()[0]
        self._in_idx = in_d["index"]; self._out_idx = out_d["index"]
        self._in_dtype  = in_d["dtype"]; self._out_dtype = out_d["dtype"]
        self._in_scale, self._in_zero   = in_d.get("quantization",  (0.0, 0))
        self._out_scale, self._out_zero = out_d.get("quantization", (0.0, 0))
        print(f"[TFLite] {os.path.basename(tflite_path)} | "
              f"in={self._in_dtype.__name__} out={self._out_dtype.__name__} "
              f"threads={n_threads}")

    def predict(self, X, verbose=0):
        outs = []
        for i in range(len(X)):
            x = X[i:i+1]
            if self._in_dtype in (np.int8, np.uint8) and self._in_scale != 0:
                x = np.clip(np.round(x / self._in_scale + self._in_zero),
                            np.iinfo(self._in_dtype).min,
                            np.iinfo(self._in_dtype).max).astype(self._in_dtype)
            else:
                x = x.astype(self._in_dtype)
            self.interpreter.set_tensor(self._in_idx, x)
            self.interpreter.invoke()
            o = self.interpreter.get_tensor(self._out_idx).copy()
            if self._out_dtype in (np.int8, np.uint8) and self._out_scale != 0:
                o = (o.astype(np.float32) - self._out_zero) * self._out_scale
            outs.append(o)
        return np.vstack(outs).astype(np.float32)


# ----------------------------------------------------------------------------
# Train SVM di atas fitur QAT — supaya perbandingan adil
# ----------------------------------------------------------------------------
print("\n[QAT] Training SVM di atas fitur QAT-INT8 …")
extractor_qat = TFLiteFeatureExtractor(qat_tflite_path)

X_tr_q = extractor_qat.predict(X_train, verbose=0)
X_va_q = extractor_qat.predict(X_valid, verbose=0)
scaler_qat = StandardScaler().fit(np.vstack([X_tr_q, X_va_q]))
X_tr_qs = scaler_qat.transform(X_tr_q)
X_va_qs = scaler_qat.transform(X_va_q)
svm_qat = SVC(kernel="rbf", C=1.0, gamma="scale",
              probability=True, random_state=42)
svm_qat.fit(np.vstack([X_tr_qs, X_va_qs]),
            np.concatenate([y_train_int, y_valid_int]))

joblib.dump(svm_qat   , "artifacts/svm_qat.pkl")
joblib.dump(scaler_qat, "artifacts/scaler_qat.pkl")

metrics_qat = evaluate_pipeline(
    extractor_qat, svm_qat, scaler_qat,
    X_test, y_test_int, class_names, label=f"QAT INT8 ({qat_mode})"
)
print_evaluation_block(metrics_qat, class_names, "QAT INT8 — TEST SET")


In [ ]:
# ============================================================================
# RESOURCE / SIZE / METRIC COMPARISON TABLES
# ============================================================================
size_orig_total = orig_size_cnn + orig_size_svm + orig_size_scaler
size_qat_kb     = get_file_size_kb(qat_tflite_path)
size_qat_total  = (size_qat_kb +
                   get_file_size_kb("artifacts/svm_qat.pkl") +
                   get_file_size_kb("artifacts/scaler_qat.pkl"))

res_df = resource_table([
    ("Original (.keras)",
     size_orig_total, model_orig.count_params(),
     metrics_orig["feat_time"], metrics_orig["inf_time"]),
    ("QAT INT8 (.tflite)",
     size_qat_total, qat_model.count_params(),
     metrics_qat["feat_time"], metrics_qat["inf_time"]),
])
print("\n" + "=" * 78)
print("RESOURCE COMPARISON")
print("=" * 78)
print(res_df.to_string(index=False))

size_df = pd.DataFrame([
    ["Feature Extractor", orig_size_extractor, size_qat_kb],
    ["SVM Classifier"   , orig_size_svm   , get_file_size_kb("artifacts/svm_qat.pkl"   )],
    ["Feature Scaler"   , orig_size_scaler, get_file_size_kb("artifacts/scaler_qat.pkl")],
], columns=["Komponen", "Awal (KB)", "QAT (KB)"])
size_df.loc[len(size_df)] = ["TOTAL",
                             size_df["Awal (KB)"].sum(),
                             size_df["QAT (KB)"].sum()]
print("\nFILE-LEVEL SIZE COMPARISON")
print(size_df.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

summary_df = pd.DataFrame({
    "Metric"  : ["Accuracy", "Sensitivity (mean)", "Specificity (mean)",
                 "F1-Score (mean)", "AUC (macro)"],
    "Original": [
        f'{metrics_orig["accuracy"]:.4f}',
        f'{np.mean(metrics_orig["sensitivity"]):.4f}',
        f'{np.mean(metrics_orig["specificity"]):.4f}',
        f'{np.mean(metrics_orig["f1"          ]):.4f}',
        f'{metrics_orig["auc_macro"]:.4f}',
    ],
    "QAT INT8": [
        f'{metrics_qat["accuracy"]:.4f}',
        f'{np.mean(metrics_qat["sensitivity"]):.4f}',
        f'{np.mean(metrics_qat["specificity"]):.4f}',
        f'{np.mean(metrics_qat["f1"          ]):.4f}',
        f'{metrics_qat["auc_macro"]:.4f}',
    ],
})
print("\n" + "=" * 78)
print("METRIC SUMMARY (Original vs QAT INT8)")
print("=" * 78)
print(summary_df.to_string(index=False))

# Plot confusion matrix side-by-side
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_confusion_matrix(metrics_orig, class_names, ax=axes[0])
plot_confusion_matrix(metrics_qat , class_names, ax=axes[1])
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================================
# SAVE QAT ARTEFACTS TO NEW FOLDER
# ============================================================================
from datetime import datetime
out_folder = f"saved_models_qat_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(out_folder, exist_ok=True)

shutil.copy2(qat_tflite_path           , os.path.join(out_folder, "qat_feature_extractor_int8.tflite"))
shutil.copy2("artifacts/svm_qat.pkl"   , os.path.join(out_folder, "svm_classifier_qat.pkl"))
shutil.copy2("artifacts/scaler_qat.pkl", os.path.join(out_folder, "feature_scaler_qat.pkl"))

with open(os.path.join(out_folder, "training_history_qat.json"), "w") as f:
    json.dump({k: [float(x) for x in v] for k, v in qat_history.history.items()},
              f, indent=2)

with open(os.path.join(out_folder, "model_info.txt"), "w") as f:
    f.write("Quantization-Aware Training (QAT)\n")
    f.write(f"Fine-tune epochs       : {QAT_FINETUNE_EPOCHS}\n")
    f.write(f"Fine-tune time         : {qat_finetune_time:.3f} s\n")
    f.write(f"Original test accuracy : {metrics_orig['accuracy']:.4f}\n")
    f.write(f"QAT      test accuracy : {metrics_qat ['accuracy']:.4f}\n")
    f.write(f"Mode TFLite            : {qat_mode}\n")
    f.write(f"Layer di-quantize      : Conv2D, Dense, MaxPool, Flatten\n")
    f.write(f"Layer NOT quantized    : ChannelAttention (custom layer)\n")

print(f"QAT artefacts saved to {out_folder}/")
for fn in sorted(os.listdir(out_folder)):
    fp = os.path.join(out_folder, fn)
    print(f"  {fn:<45} {get_file_size_kb(fp):>10.2f} KB")
